# TITAN AI - YOLOv11 training (Colab or local)

Runs the same steps as `src/training/train.py` and `evaluate.py`.

**On Colab:** Runtime → Change runtime type → **T4 GPU**, then run the cells top to bottom.
**Locally:** skip the Colab cell and start Jupyter from the repo root.

## 1. Colab only: get the repo and install packages

In [ ]:
import os
IN_COLAB = "COLAB_GPU" in os.environ or os.path.exists("/content")
if IN_COLAB:
    !git lfs install
    !git clone https://github.com/YOUANNA13/graduation_project_aisafesight.git
    %cd graduation_project_aisafesight
    !git lfs pull
    !pip -q install ultralytics pillow

## 2. Check the GPU

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

## 3. Build the training set
Rebuilds `prepared/titan_v1/` from the repo (it is not in git). Read the report: the class table
shows how many boxes of each class are in train and val, and the WARNING line lists classes with no val examples.

In [ ]:
!python tools/prepare_dataset.py remap
!python tools/prepare_dataset.py build --pin framing=train

## 4. Train

In [ ]:
from ultralytics import YOLO

MODEL = "yolo11s.pt"   # yolo11n.pt = faster/lighter, yolo11m.pt = bigger (needs more GPU memory)
IMGSZ = 960
NAME = f"v1_{MODEL[4:-3]}_{IMGSZ}"

model = YOLO(MODEL)
model.train(
    data="prepared/titan_v1/data.yaml",
    imgsz=IMGSZ, epochs=100, patience=25,
    batch=-1,                 # auto batch size for this GPU
    workers=2,
    cos_lr=True, close_mosaic=10, amp=True,
    seed=42, deterministic=True,
    project="runs/titan", name=NAME, plots=True,
)
BEST = f"runs/titan/{NAME}/weights/best.pt"

## 5. Per-class evaluation (worst classes first)

In [ ]:
best = YOLO(BEST)
m = best.val(data="prepared/titan_v1/data.yaml", imgsz=IMGSZ, split="val", plots=True)
rows = []
for k, c in enumerate(m.box.ap_class_index):
    p, r, ap50, ap = m.box.class_result(k)
    rows.append((best.names[int(c)], p, r, ap50, ap))
print(f"{'class':<15}{'P':>7}{'R':>7}{'mAP50':>8}{'mAP50-95':>10}")
for n, p, r, ap50, ap in sorted(rows, key=lambda x: x[3]):
    print(f"{n:<15}{p:>7.3f}{r:>7.3f}{ap50:>8.3f}{ap:>10.3f}")
print(f"{'ALL':<15}{m.box.mp:>7.3f}{m.box.mr:>7.3f}{m.box.map50:>8.3f}{m.box.map:>10.3f}")

In [ ]:
from IPython.display import Image, display
display(Image(filename=f"{m.save_dir}/confusion_matrix_normalized.png", width=700))

## 6. Predict on the overhead video
This video is in the training set, so it only shows that training worked. Use the Hikvision test video for a real test.

In [ ]:
for r in best.predict("raw/framing/original_video.mp4", imgsz=IMGSZ, conf=0.25,
                      save=True, stream=True, project="runs/titan", name=f"predict_{NAME}"):
    pass
print("Saved to", r.save_dir)

## 7. Colab only: download the weights

In [ ]:
if IN_COLAB:
    from google.colab import files
    !zip -qr {NAME}.zip runs/titan/{NAME}
    files.download(f"{NAME}.zip")